# 2026 COMP90042 Project
*Make sure you change the file name with your group id.*

# Readme
*If there is something to be noted for the marker, please mention here.*

*If you are planning to implement a program with Object Oriented Programming style, please put those the bottom of this ipynb file*

# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [25]:
# Install required packages, including rank_bm25 for BM25-based retrieval.
!pip install rank_bm25 transformers accelerate -q

In [26]:
# Import necessary packages and modules.
import os
import json
import re
import random
import numpy as np
from tqdm import tqdm

import torch
from torch.utils.data import Dataset

from rank_bm25 import BM25Okapi
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [27]:
# =========================
# File Paths
# =========================
DATA_DIR = "/content"

train_path = os.path.join(DATA_DIR, "train-claims.json")
dev_path = os.path.join(DATA_DIR, "dev-claims.json")
test_path = os.path.join(DATA_DIR, "test-claims-unlabelled.json")
baseline_path = os.path.join(DATA_DIR, "dev-claims-baseline.json")
evidence_path = os.path.join(DATA_DIR, "evidence.json")


# =========================
# Generic JSON Loader
# =========================
def load_json(path):
    """
    Load any JSON file from disk.

    Args:
        path (str): file path

    Returns:
        dict: loaded JSON content
    """
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


# =========================
# Load All Project Data
# =========================
print("Loading datasets...")

train_claims = load_json(train_path)
dev_claims = load_json(dev_path)
test_claims = load_json(test_path)
baseline_claims = load_json(baseline_path)
evidence_dict = load_json(evidence_path)


# =========================
# Basic Statistics
# =========================
print("Train claims:", len(train_claims))
print("Dev claims:", len(dev_claims))
print("Test claims:", len(test_claims))
print("Baseline predictions:", len(baseline_claims))
print("Evidence passages:", len(evidence_dict))


# =========================
# Preview Evidence Samples
# =========================
print("\nSample evidence examples:")

for i, (eid, text) in enumerate(evidence_dict.items()):
    print(f"{eid}: {text[:200]}")
    if i >= 2:
        break

Loading datasets...
Train claims: 1228
Dev claims: 154
Test claims: 153
Baseline predictions: 154
Evidence passages: 1208827

Sample evidence examples:
evidence-0: John Bennet Lawes, English entrepreneur and agricultural scientist
evidence-1: Lindberg began his professional career at the age of 16, eventually moving to New York City in 1977.
evidence-2: ``Boston (Ladies of Cambridge)'' by Vampire Weekend


In [28]:
import re
import numpy as np
from tqdm import tqdm
from rank_bm25 import BM25Okapi

def simple_tokenize(text):
    """
    Tokenizer for BM25.
    Lowercase text and keep only words/numbers.
    """
    text = text.lower()
    tokens = re.findall(r"[a-z0-9]+", text)
    return tokens

# Acquire evidence id and text
evidence_ids = list(evidence_dict.keys())
evidence_texts = list(evidence_dict.values())

print("Tokenizing evidence corpus...")

# Tokenize each evidence document
tokenized_evidence = [
    simple_tokenize(text)
    for text in tqdm(evidence_texts)
]

print("Building BM25 index...")

# Build BM25 model over the tokenized corpus
# This object will be used to compute relevance scores between a query and documents
bm25 = BM25Okapi(tokenized_evidence)

print("BM25 index built.")

Tokenizing evidence corpus...


100%|██████████| 1208827/1208827 [00:49<00:00, 24660.85it/s]


Building BM25 index...
BM25 index built.


In [29]:
# Mapping from label names to integer IDs
# This is used to convert string labels into numerical format for model training
label2id = {
    "SUPPORTS": 0,
    "REFUTES": 1,
    "NOT_ENOUGH_INFO": 2,
    "DISPUTED": 3
}

# Reverse mapping from IDs back to label names
# This is useful for converting model predictions (integers) back to human-readable labels
id2label = {v: k for k, v in label2id.items()}

In [30]:
def build_examples(claims_dict, labelled=True):
    """
    Build claim-level examples.

    For train/dev:
        Use provided gold evidence IDs.

    For test:
        No labels and often no gold evidence,
        so fallback to BM25 retrieval.
    """
    examples = []

    for claim_id, item in tqdm(claims_dict.items()):
        claim_text = item["claim_text"]

        # ==========================================
        # If gold evidences exist (train/dev)
        # ==========================================
        if "evidences" in item:
            evidence_ids_for_claim = item["evidences"][:5]

            evidence_texts_for_claim = [
                evidence_dict[eid] for eid in evidence_ids_for_claim
            ]

        # ==========================================
        # If no evidences exist (test set)
        # ==========================================
        else:
            evidence_ids_for_claim, evidence_texts_for_claim = retrieve_top_k_evidence(
                claim_text,
                top_k=5
            )

        example = {
            "claim_id": claim_id,
            "claim_text": claim_text,
            "evidence_ids": evidence_ids_for_claim,
            "evidence_texts": evidence_texts_for_claim,
            "evidence_text": " [SEP] ".join(evidence_texts_for_claim)
        }

        if labelled and "claim_label" in item:
            example["label"] = label2id[item["claim_label"]]

        examples.append(example)

    return examples

In [31]:
SMALL_TRAIN_SIZE = 300
SMALL_DEV_SIZE = 50
# SMALL_TEST_SIZE = 50

train_claims_small = dict(list(train_claims.items()))
dev_claims_small = dict(list(dev_claims.items()))
# test_claims_small = dict(list(test_claims.items())[:SMALL_TEST_SIZE])

train_examples = build_examples(
    train_claims_small,
    labelled=True
)

dev_examples = build_examples(
    dev_claims_small,
    labelled=True
)

# test_examples = build_examples(
#    test_claims_small,
#    labelled=False
# )

print("Train examples:", len(train_examples))
print("Dev examples:", len(dev_examples))
# print("Test examples:", len(test_examples))

print("\nSample processed example:")
print(train_examples[0])

100%|██████████| 154/154 [00:00<00:00, 134511.21it/s]

Train examples: 1228
Dev examples: 154

Sample processed example:
{'claim_id': 'claim-1937', 'claim_text': 'Not only is there no scientific evidence that CO2 is a pollutant, higher CO2 concentrations actually help ecosystems support more plant and animal life.', 'evidence_ids': ['evidence-442946', 'evidence-1194317', 'evidence-12171'], 'evidence_texts': ['At very high concentrations (100 times atmospheric concentration, or greater), carbon dioxide can be toxic to animal life, so raising the concentration to 10,000 ppm (1%) or higher for several hours will eliminate pests such as whiteflies and spider mites in a greenhouse.', 'Plants can grow as much as 50 percent faster in concentrations of 1,000 ppm CO 2 when compared with ambient conditions, though this assumes no change in climate and no limitation on other nutrients.', 'Higher carbon dioxide concentrations will favourably affect plant growth and demand for water.'], 'evidence_text': 'At very high concentrations (100 times atmospher

# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)
## **Fine-tuned BERT**



## Micro-level 分类模型训练流程

本部分实现 sentence-level(micro-level)分类模型,用于判断单条evidence对claim的支持关系。

---

### Step 1: 构造 micro-level 数据

将 claim-level 数据转换为 sentence-level:

- 原始: claim →多条 evidence → 一个 label

- 转换后：(claim, evidence_i) → label

说明：
- 跳过 DISPUTED(不适合单句建模)
- 每条 evidence 继承 claim 的标签

---

### Step 2: 类别不平衡处理

使用类别权重:$$w_i = N / (C * n_i)$$
其中: N—总样本数, C—类别数, n_i—第i类样本数

作用：减少模型偏向多数类

---

### Step 3：构建输入（BERT格式）

输入形式：[CLS] claim [SEP] evidence [SEP]

模型学习：
该 evidence 是否支持/反驳/无法判断该 claim

---

### Step 4：模型

使用 BERT 分类模型：

AutoModelForSequenceClassification

输出：
3 分类（SUPPORTS / REFUTES / NOT_ENOUGH_INFO）

---

## Step 5：损失函数

使用加权交叉熵：

$$Loss = - w_y log p(y)$$

作用：
- 提高少数类权重

---

## Step 6：训练

流程：

输入 → BERT → logits  
→ 计算 loss  
→ 反向传播  
→ 更新参数

---

## Step 7：评估

使用指标：
- Accuracy
- Macro F1
- Weighted F1

说明：
- Macro F1 更适合不平衡数据

In [32]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LENGTH = 256
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
EPOCHS = 1

micro_label2id = {
    "SUPPORTS": 0,
    "REFUTES": 1,
    "NOT_ENOUGH_INFO": 2
}

micro_id2label = {v: k for k, v in micro_label2id.items()}


def build_micro_examples(examples, labelled=True, skip_disputed=True):
    """
    Convert claim-level examples into sentence-level micro examples.

    Each claim with provided evidence becomes multiple micro examples:
        claim + evidence_i -> micro label

    DISPUTED is skipped during micro training because it is a macro-level label.
    """
    micro_examples = []

    for ex in examples:
        claim_id = ex["claim_id"]
        claim_text = ex["claim_text"]

        evidence_ids = ex["evidence_ids"]
        evidence_texts = ex["evidence_texts"]

        if labelled:
            macro_label = id2label[ex["label"]]

            if skip_disputed and macro_label == "DISPUTED":
                continue

            if macro_label not in micro_label2id:
                continue

            micro_label = micro_label2id[macro_label]

        for rank, (eid, ev_text) in enumerate(zip(evidence_ids, evidence_texts)):
            item = {
                "claim_id": claim_id,
                "claim_text": claim_text,
                "evidence_id": eid,
                "evidence_text": ev_text,
                "rank": rank
            }

            if labelled:
                item["label"] = micro_label

            micro_examples.append(item)

    return micro_examples


train_micro_examples = build_micro_examples(
    train_examples,
    labelled=True,
    skip_disputed=True
)

dev_micro_examples = build_micro_examples(
    dev_examples,
    labelled=True,
    skip_disputed=True
)

print("Train micro examples:", len(train_micro_examples))
print("Dev micro examples:", len(dev_micro_examples))
print(train_micro_examples[0])

Train micro examples: 3730
Dev micro examples: 433
{'claim_id': 'claim-126', 'claim_text': 'El Niño drove record highs in global temperatures suggesting rise may not be down to man-made emissions.', 'evidence_id': 'evidence-338219', 'evidence_text': 'While ‘climate change’ can be due to natural forces or human activity, there is now substantial evidence to indicate that human activity – and specifically increased greenhouse gas (GHGs) emissions – is a key factor in the pace and extent of global temperature increases.', 'rank': 0, 'label': 1}


In [33]:
from collections import Counter
import torch
import numpy as np

train_label_counts = Counter([ex["label"] for ex in train_micro_examples])

num_classes = len(micro_label2id)
total_count = sum(train_label_counts.values())

class_weights = []

for i in range(num_classes):
    weight = total_count / (num_classes * train_label_counts[i])
    class_weights.append(weight)

class_weights = torch.tensor(class_weights, dtype=torch.float)

print("Class weights:", class_weights)

Class weights: tensor([0.9258, 2.7206, 0.6442])


In [34]:
class MicroClaimEvidenceDataset(Dataset):
    """
    Dataset for sentence-level BERT micro-verdict prediction.

    Input:
        claim_text + one evidence sentence

    Output:
        SUPPORTS / REFUTES / NOT_ENOUGH_INFO
    """

    def __init__(self, micro_examples, tokenizer, max_length=256, labelled=True):
        self.examples = micro_examples
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.labelled = labelled

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]

        encoding = self.tokenizer(
            ex["claim_text"],
            ex["evidence_text"],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
        }

        if "token_type_ids" in encoding:
            item["token_type_ids"] = encoding["token_type_ids"].squeeze(0)

        if self.labelled:
            item["labels"] = torch.tensor(ex["label"], dtype=torch.long)

        return item
train_micro_dataset = MicroClaimEvidenceDataset(
    train_micro_examples,
    tokenizer,
    max_length=MAX_LENGTH,
    labelled=True
)

dev_micro_dataset = MicroClaimEvidenceDataset(
    dev_micro_examples,
    tokenizer,
    max_length=MAX_LENGTH,
    labelled=True
)

print("Train micro dataset:", len(train_micro_dataset))
print("Dev micro dataset:", len(dev_micro_dataset))

Train micro dataset: 3730
Dev micro dataset: 433


In [35]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=micro_id2label,
    label2id=micro_label2id
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [36]:
from transformers import Trainer
import torch.nn as nn

class WeightedLossTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)

        logits = outputs.logits

        loss_fn = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device)
        )

        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [37]:
from sklearn.metrics import accuracy_score, f1_score

def compute_micro_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    weighted_f1 = f1_score(labels, preds, average="weighted")

    return {
        "micro_accuracy": accuracy,
        "micro_macro_f1": macro_f1,
        "micro_weighted_f1": weighted_f1
    }

In [38]:
training_args = TrainingArguments(
    output_dir="./micro_macro_bert",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,

    logging_dir="./logs",
    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="micro_macro_f1",#"micro_accuracy",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none"
)
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_micro_dataset,
    eval_dataset=dev_micro_dataset,
    compute_metrics=compute_micro_metrics,
    class_weights=class_weights
)
trainer.train()
micro_dev_results = trainer.evaluate()
print(micro_dev_results)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Micro Accuracy,Micro Macro F1,Micro Weighted F1
1,0.824215,0.952711,0.575058,0.528676,0.571231


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

{'eval_loss': 0.9527110457420349, 'eval_micro_accuracy': 0.5750577367205543, 'eval_micro_macro_f1': 0.5286757348234934, 'eval_micro_weighted_f1': 0.5712307962493769, 'eval_runtime': 7.757, 'eval_samples_per_second': 55.821, 'eval_steps_per_second': 7.09, 'epoch': 1.0}


##  Macro-level 聚合模型训练流程

本部分实现 claim-level（macro-level）分类模型，用于将多条 evidence 的 micro 预测结果聚合为最终标签。

---

### Step 1：定义 Macro 标签

Macro-level 为 4 分类：

- SUPPORTS
- REFUTES
- NOT_ENOUGH_INFO
- DISPUTED

说明：
- DISPUTED 只在 macro-level 判断（需要多条 evidence 之间的冲突信息）

---

### Step 2：获取 Micro 预测概率

使用训练好的 micro BERT，对每条 evidence 预测：

(claim, evidence_i) → [P(SUPPORTS), P(REFUTES), P(NEI)]

得到：

$$ micro_probs ∈ R^{num_evidence × 3}$$

---

### Step 3：构造不确定性特征

对每个 claim 计算两个特征：

- JSD（Jensen-Shannon Divergence）
- support_refute_variance

作用：

- JSD：衡量不同 evidence 预测分布之间的差异
- variance：衡量 SUPPORTS 和 REFUTES 的波动程度

用于辅助判断 DISPUTED

---

### Step 4：构建 Macro Features

每个 claim 转换为：

{
  claim_id,
  evidence_ids,
  micro_probs,
  uncertainty_features,
  label
}

---

### Step 5：统一 Evidence 数量（Padding / Truncation）

由于每个 claim 的 evidence 数量不同：

- 多于 max_evidence → 截断
- 少于 max_evidence → padding（补0）

并构造：

evidence_mask ∈ {0,1}

最终输入：

micro_probs: [max_evidence, 3]  
evidence_mask: [max_evidence]

---

### Step 6：Attention 聚合

为每条 evidence 计算权重：

$$attention_weight_i = softmax(attention(micro_prob_i))$$

加权求和：

$$weighted_probs = Σ (attention_weight_i × micro_prob_i)$$

作用：

- 强调重要 evidence
- 降低噪声 evidence 的影响

---

### Step 7：Macro 分类

拼接特征：

[weighted_probs, uncertainty_features]

输入分类器：

→ $$logits ∈ R^4$$
→ 预测 macro label

---

### Step 8：训练与评估

训练：

输入 → attention → 聚合 → 分类器  
→ CrossEntropyLoss  
→ 反向传播更新参数

评估：

- Dev Accuracy

---
### 总结
- 聚合多条 evidence 的 micro 预测
- 利用 attention 学习重要性权重
- 引入不确定性特征

In [39]:
macro_label2id = {
    "SUPPORTS": 0,
    "REFUTES": 1,
    "NOT_ENOUGH_INFO": 2,
    "DISPUTED": 3
}

macro_id2label = {v: k for k, v in macro_label2id.items()}

def entropy_np(probs):
    probs = np.array(probs, dtype=np.float32)
    return -np.sum(probs * np.log(probs + 1e-8))


def js_divergence_np(prob_list):
    prob_array = np.array(prob_list, dtype=np.float32)
    mean_prob = prob_array.mean(axis=0)
    return entropy_np(mean_prob) - np.mean([entropy_np(p) for p in prob_array])


def support_refute_variance_np(prob_list):
    prob_array = np.array(prob_list, dtype=np.float32)
    return np.var(prob_array[:, 0]) + np.var(prob_array[:, 1])
def get_micro_probs_for_claim(claim_text, evidence_texts):
    """
    Use trained micro BERT to predict probs for all evidences of one claim.
    Output shape: [num_evidence, 3]
    """
    model.eval()
    all_probs = []

    for ev_text in evidence_texts:
        encoding = tokenizer(
            claim_text,
            ev_text,
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        encoding = {k: v.to(model.device) for k, v in encoding.items()}

        with torch.no_grad():
            outputs = model(**encoding)
            temperature = 1
            probs = torch.softmax(outputs.logits/temperature, dim=-1).squeeze(0)

        all_probs.append(probs.cpu().numpy())

    return np.array(all_probs, dtype=np.float32)
def build_macro_features(examples, labelled=True):
    """
    Build claim-level features for learned attention aggregation.

    Each claim becomes:
        micro_probs: [num_evidence, 3]
        uncertainty_features: [JSD, support_refute_variance]
        label: 4-class macro label
    """
    feature_examples = []

    for ex in examples:
        claim_text = ex["claim_text"]
        evidence_texts = ex["evidence_texts"]

        micro_probs = get_micro_probs_for_claim(claim_text, evidence_texts)

        jsd = js_divergence_np(micro_probs)
        sr_var = support_refute_variance_np(micro_probs)

        item = {
            "claim_id": ex["claim_id"],
            "evidence_ids": ex["evidence_ids"],
            "micro_probs": micro_probs,
            "uncertainty_features": np.array([jsd, sr_var], dtype=np.float32)
        }

        if labelled:
            macro_label = id2label[ex["label"]]
            item["label"] = macro_label2id[macro_label]

        feature_examples.append(item)

    return feature_examples
train_macro_features = build_macro_features(train_examples, labelled=True)
dev_macro_features = build_macro_features(dev_examples, labelled=True)

print("Train macro features:", len(train_macro_features))
print("Dev macro features:", len(dev_macro_features))
print(train_macro_features[0].keys())

Train macro features: 1228
Dev macro features: 154
dict_keys(['claim_id', 'evidence_ids', 'micro_probs', 'uncertainty_features', 'label'])


In [40]:
from torch.utils.data import DataLoader
class MacroAggregationDataset(Dataset):
    def __init__(self, feature_examples, labelled=True, max_evidence=5):
        self.examples = feature_examples
        self.labelled = labelled
        self.max_evidence = max_evidence

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]

        micro_probs = torch.tensor(ex["micro_probs"], dtype=torch.float32)

        # pad / truncate to fixed number of evidence
        num_evidence = micro_probs.shape[0]

        if num_evidence > self.max_evidence:
            micro_probs = micro_probs[:self.max_evidence]
            evidence_mask = torch.ones(self.max_evidence, dtype=torch.float32)

        else:
            pad_size = self.max_evidence - num_evidence
            pad = torch.zeros(pad_size, micro_probs.shape[1])
            micro_probs = torch.cat([micro_probs, pad], dim=0)

            evidence_mask = torch.cat([
                torch.ones(num_evidence, dtype=torch.float32),
                torch.zeros(pad_size, dtype=torch.float32)
            ])

        item = {
            "claim_id": ex["claim_id"],
            "evidence_ids": ex["evidence_ids"],
            "micro_probs": micro_probs,
            "evidence_mask": evidence_mask,
            "uncertainty_features": torch.tensor(ex["uncertainty_features"], dtype=torch.float32)
        }

        if self.labelled:
            item["labels"] = torch.tensor(ex["label"], dtype=torch.long)

        return item
def macro_collate_fn(batch):
    output = {
        "claim_ids": [x["claim_id"] for x in batch],
        "evidence_ids": [x["evidence_ids"] for x in batch],
        "micro_probs": torch.stack([x["micro_probs"] for x in batch]),
        "evidence_mask": torch.stack([x["evidence_mask"] for x in batch]),
        "uncertainty_features": torch.stack([x["uncertainty_features"] for x in batch]),
    }

    if "labels" in batch[0]:
        output["labels"] = torch.stack([x["labels"] for x in batch])

    return output

train_macro_dataset = MacroAggregationDataset(train_macro_features, labelled=True)
dev_macro_dataset = MacroAggregationDataset(dev_macro_features, labelled=True)

train_macro_loader = DataLoader(
    train_macro_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=macro_collate_fn
)

dev_macro_loader = DataLoader(
    dev_macro_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=macro_collate_fn
)

In [41]:
import torch.nn as nn
class LearnedAttentionMacroAggregator(nn.Module):
    def __init__(self, num_micro_labels=3, hidden_size=32, num_macro_labels=4):
        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(num_micro_labels, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(num_micro_labels + 2, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_macro_labels)
        )

    def forward(self, micro_probs, uncertainty_features, evidence_mask=None, labels=None):
        attn_logits = self.attention(micro_probs).squeeze(-1)

        if evidence_mask is not None:
            attn_logits = attn_logits.masked_fill(evidence_mask == 0, -1e9)

        attn_weights = torch.softmax(attn_logits, dim=1)

        weighted_probs = torch.sum(
            micro_probs * attn_weights.unsqueeze(-1),
            dim=1
        )

        features = torch.cat([weighted_probs, uncertainty_features], dim=-1)
        logits = self.classifier(features)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return {
            "loss": loss,
            "logits": logits,
            "attention_weights": attn_weights
        }
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

macro_model = LearnedAttentionMacroAggregator().to(device)

optimizer = torch.optim.AdamW(
    macro_model.parameters(),
    lr=1e-3,
    weight_decay=0.01
)

In [42]:
def train_macro_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0

    for batch in loader:
        optimizer.zero_grad()

        micro_probs = batch["micro_probs"].to(device)
        uncertainty_features = batch["uncertainty_features"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            micro_probs=micro_probs,
            uncertainty_features=uncertainty_features,
            labels=labels
        )

        loss = outputs["loss"]
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate_macro_model(model, loader, device):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            micro_probs = batch["micro_probs"].to(device)
            uncertainty_features = batch["uncertainty_features"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                micro_probs=micro_probs,
                uncertainty_features=uncertainty_features
            )

            preds = torch.argmax(outputs["logits"], dim=-1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return np.mean(np.array(all_preds) == np.array(all_labels))

MACRO_EPOCHS = 20

for epoch in range(MACRO_EPOCHS):
    train_loss = train_macro_one_epoch(
        macro_model,
        train_macro_loader,
        optimizer,
        device
    )

    dev_acc = evaluate_macro_model(
        macro_model,
        dev_macro_loader,
        device
    )

    print(f"Epoch {epoch + 1}/{MACRO_EPOCHS}")
    print("Macro train loss:", train_loss)
    print("Macro dev accuracy:", dev_acc)

Epoch 1/20
Macro train loss: 1.266853949466309
Macro dev accuracy: 0.5
Epoch 2/20
Macro train loss: 1.0998684551034654
Macro dev accuracy: 0.6038961038961039
Epoch 3/20
Macro train loss: 0.9549270427072203
Macro dev accuracy: 0.6298701298701299
Epoch 4/20
Macro train loss: 0.828492879286989
Macro dev accuracy: 0.6428571428571429
Epoch 5/20
Macro train loss: 0.7598650830906707
Macro dev accuracy: 0.6428571428571429
Epoch 6/20
Macro train loss: 0.7094409243239985
Macro dev accuracy: 0.6428571428571429
Epoch 7/20
Macro train loss: 0.6964527809774721
Macro dev accuracy: 0.6428571428571429
Epoch 8/20
Macro train loss: 0.6711428891141693
Macro dev accuracy: 0.6493506493506493
Epoch 9/20
Macro train loss: 0.6539474228372821
Macro dev accuracy: 0.6428571428571429
Epoch 10/20
Macro train loss: 0.6420685629565994
Macro dev accuracy: 0.6363636363636364
Epoch 11/20
Macro train loss: 0.6322250413623723
Macro dev accuracy: 0.6298701298701299
Epoch 12/20
Macro train loss: 0.6160462621551055
Macro dev

In [43]:
def predict_macro_with_learned_attention(model, loader, device):
    model.eval()
    predictions = {}

    with torch.no_grad():
        for batch in loader:
            micro_probs = batch["micro_probs"].to(device)
            uncertainty_features = batch["uncertainty_features"].to(device)

            outputs = model(
                micro_probs=micro_probs,
                uncertainty_features=uncertainty_features
            )

            preds = torch.argmax(outputs["logits"], dim=-1).cpu().numpy()

            for claim_id, evidence_ids, pred_id in zip(
                batch["claim_ids"],
                batch["evidence_ids"],
                preds
            ):
                predictions[claim_id] = {
                    "claim_label": macro_id2label[int(pred_id)],
                    "evidences": evidence_ids
                }

    return predictions

import json

dev_predictions = predict_macro_with_learned_attention(
    macro_model,
    dev_macro_loader,
    device
)

with open("dev_predictions_learned_attention_macro.json", "w", encoding="utf-8") as f:
    json.dump(dev_predictions, f, indent=2)

with open("dev_claims_small.json", "w", encoding="utf-8") as f:
    json.dump(dev_claims_small, f, indent=2)

!python eval.py \
    --predictions dev_predictions_learned_attention_macro.json \
    --groundtruth dev_claims_small.json

Evidence Retrieval F-score (F)    = 1.0
Claim Classification Accuracy (A) = 0.6623376623376623
Harmonic Mean of F and A          = 0.796875


In [44]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

def evaluate_micro_per_class(model, dataset, tokenizer=None):
    model.eval()

    all_preds = []
    all_labels = []

    for i in range(len(dataset)):
        item = dataset[i]

        inputs = {
            "input_ids": item["input_ids"].unsqueeze(0).to(model.device),
            "attention_mask": item["attention_mask"].unsqueeze(0).to(model.device),
        }

        if "token_type_ids" in item:
            inputs["token_type_ids"] = item["token_type_ids"].unsqueeze(0).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs)
            pred = torch.argmax(outputs.logits, dim=-1).item()

        all_preds.append(pred)
        all_labels.append(item["labels"].item())

    print("Micro-level classification report:")
    print(
        classification_report(
            all_labels,
            all_preds,
            target_names=[
                "SUPPORTS",
                "REFUTES",
                "NOT_ENOUGH_INFO"
            ],
            digits=4
        )
    )

    print("Confusion matrix:")
    print(confusion_matrix(all_labels, all_preds))

    return all_labels, all_preds
micro_true, micro_pred = evaluate_micro_per_class(
    model,
    dev_micro_dataset
)

Micro-level classification report:
                 precision    recall  f1-score   support

       SUPPORTS     0.5455    0.7018    0.6138       171
        REFUTES     0.4255    0.3509    0.3846        57
NOT_ENOUGH_INFO     0.6566    0.5317    0.5876       205

       accuracy                         0.5751       433
      macro avg     0.5425    0.5281    0.5287       433
   weighted avg     0.5823    0.5751    0.5712       433

Confusion matrix:
[[120   8  43]
 [ 23  20  14]
 [ 77  19 109]]


In [45]:
from sklearn.metrics import classification_report, confusion_matrix

def evaluate_macro_predictions(predictions, groundtruth_claims):
    y_true = []
    y_pred = []

    for claim_id, pred_item in predictions.items():
        if claim_id not in groundtruth_claims:
            continue

        true_label = groundtruth_claims[claim_id]["claim_label"]
        pred_label = pred_item["claim_label"]

        y_true.append(true_label)
        y_pred.append(pred_label)

    labels = ["SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO", "DISPUTED"]

    print("Macro-level classification report:")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=labels,
            target_names=labels,
            digits=4
        )
    )

    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred, labels=labels))

    return y_true, y_pred
macro_true, macro_pred = evaluate_macro_predictions(
    dev_predictions,
    dev_claims
)

Macro-level classification report:
                 precision    recall  f1-score   support

       SUPPORTS     0.5962    0.9118    0.7209        68
        REFUTES     0.6667    0.2963    0.4103        27
NOT_ENOUGH_INFO     0.8649    0.7805    0.8205        41
       DISPUTED     0.0000    0.0000    0.0000        18

       accuracy                         0.6623       154
      macro avg     0.5319    0.4971    0.4879       154
   weighted avg     0.6104    0.6623    0.6087       154

Confusion matrix:
[[62  3  3  0]
 [17  8  1  1]
 [ 9  0 32  0]
 [16  1  1  0]]


# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## Object Oriented Programming codes here

*You can use multiple code snippets. Just add more if needed*